# CUDA #2 — Pierwsze testy: kompilacja, wykrywanie GPU, pamięć, strumienie i timingi

W tym notatniku przejdziemy przez zestaw **praktycznych testów CUDA**, które możesz umieścić w drugim wpisie serii:

1. Test kompilacji NVCC (czy `.cu` się kompilują)
2. Wykrywanie GPU i odczyt parametrów
3. Makro do obsługi błędów i sprawdzanie błędów po kernelu
4. Minimalny kernel `Hello from GPU`
5. Testy pamięci: `cudaMalloc` i `cudaMemcpy`
6. Strumienie CUDA (krótka zajawka)
7. Pomiar czasu z `cudaEvent`
8. Wersje runtime/drivera
9. Unified Memory (Managed) — test podstawowy

> **Uwaga**: Notatnik zakłada, że **CUDA Toolkit** (w tym `nvcc`) oraz sterowniki NVIDIA są zainstalowane, a środowisko ma dostęp do GPU. Kod kompilujemy i uruchamiamy poleceniami powłoki (Jupyter magic `!` / `%%bash`).


## 0) Wymagania i szybka diagnostyka środowiska
Poniższe komórki sprawdzają dostępność kompilatora i GPU.


In [ ]:
!nvcc --version || echo 'nvcc nie znaleziono (sprawdź instalację CUDA Toolkit)'
!nvidia-smi || echo 'nvidia-smi nie znaleziono (brak sterownika lub brak uprawnień)'

## 1) Test kompilacji NVCC (host-only)
Celem jest potwierdzenie, że `nvcc` poprawnie kompiluje prosty plik `.cu`.


In [ ]:
%%writefile hello_nvcc.cu
#include <cstdio>

int main(){
    std::puts("OK: NVCC kompiluje .cu (host-only).");
    return 0;
}

In [ ]:
!nvcc -O2 -o hello_nvcc hello_nvcc.cu && ./hello_nvcc


## 2) Wykrywanie GPU i właściwości urządzenia
Pobieramy liczbę urządzeń oraz kluczowe parametry z `cudaGetDeviceProperties`.


In [ ]:

%%writefile device_query.cu
#include <cstdio>
#include <cuda_runtime.h>

#define CUDA_CHECK(x) do {     cudaError_t err__ = (x);     if (err__ != cudaSuccess) {         std::fprintf(stderr, "CUDA error %s at %s:%d
", cudaGetErrorString(err__), __FILE__, __LINE__);         std::exit(1);     } } while(0)

int main(){
    int count = 0;
    CUDA_CHECK(cudaGetDeviceCount(&count));
    std::printf("Liczba urządzeń: %d", count);
    for (int d = 0; d < count; ++d) {
        cudaDeviceProp p{};
        CUDA_CHECK(cudaGetDeviceProperties(&p, d));
        std::printf("
#%d: %s
", d, p.name);
        std::printf("  Compute Capability: %d.%d
", p.major, p.minor);
        std::printf("  SMs: %d
", p.multiProcessorCount);
        std::printf("  GlobalMem: %.2f GB
", p.totalGlobalMem / (1024.0*1024.0*1024.0));
        std::printf("  SharedMem/Block: %zu KB
", p.sharedMemPerBlock / 1024UL);
        std::printf("  WarpSize: %d
", p.warpSize);
        std::printf("  MaxThreads/Block: %d
", p.maxThreadsPerBlock);
        std::printf("  MaxThreadsDim: (%d,%d,%d)
", p.maxThreadsDim[0], p.maxThreadsDim[1], p.maxThreadsDim[2]);
        std::printf("  MaxGridSize:   (%d,%d,%d)
", p.maxGridSize[0], p.maxGridSize[1], p.maxGridSize[2]);
        std::printf("  ClockRate: %.2f MHz
", p.clockRate / 1000.0);
    }
    return 0;
}


In [ ]:
!nvcc -O2 device_query.cu -o device_query && ./device_query


## 3) Obsługa błędów + sprawdzanie błędów po kernelu
CUDA nie rzuca wyjątków — należy sprawdzać kody błędów. Poniżej makro `CUDA_CHECK` oraz przykład wymuszenia błędu (`cudaSetDevice(999)`).


In [ ]:

%%writefile error_check_demo.cu
#include <cstdio>
#include <cuda_runtime.h>

#define CUDA_CHECK(x) do {     cudaError_t err__ = (x);     if (err__ != cudaSuccess) {         std::fprintf(stderr, "CUDA error %s at %s:%d
", cudaGetErrorString(err__), __FILE__, __LINE__);         std::exit(1);     } } while(0)

__global__ void noop() { /* nic */ }

int main(){
    // Wymuszony błąd — zwykle nie masz 1000 urządzeń.
    cudaError_t e = cudaSetDevice(999);
    if (e != cudaSuccess) {
        std::fprintf(stderr, "Oczekiwany błąd: %s
", cudaGetErrorString(e));
    }

    // Poprawne użycie: ustaw urządzenie 0 (jeśli istnieje)
    int count = 0; CUDA_CHECK(cudaGetDeviceCount(&count));
    if (count == 0) { std::puts("Brak GPU"); return 0; }
    CUDA_CHECK(cudaSetDevice(0));

    // Uruchom kernel i sprawdź błąd bezpośrednio po wywołaniu
    noop<<<1, 1>>>();
    CUDA_CHECK(cudaGetLastError()); // błąd konfiguracji, jeśli jakiś wystąpił
    CUDA_CHECK(cudaDeviceSynchronize());
    std::puts("OK: kernel wykonany, błędy niewykryte.");
    return 0;
}


In [ ]:
!nvcc -O2 error_check_demo.cu -o error_check_demo && ./error_check_demo


## 4) Minimalny kernel: `Hello from GPU`
Demonstracja konfiguracji `<<<grid, block>>>` oraz `printf` w kernelu.


In [ ]:

%%writefile hello_kernel.cu
#include <cstdio>
#include <cuda_runtime.h>

#define CUDA_CHECK(x) do {     cudaError_t err__ = (x);     if (err__ != cudaSuccess) {         std::fprintf(stderr, "CUDA error %s at %s:%d
", cudaGetErrorString(err__), __FILE__, __LINE__);         std::exit(1);     } } while(0)

__global__ void hello(){
    printf("Hello from GPU! block %d, thread %d (blockDim.x=%d)
", blockIdx.x, threadIdx.x, blockDim.x);
}

int main(){
    dim3 grid(2), block(4);
    hello<<<grid, block>>>();
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());
    return 0;
}


In [ ]:
!nvcc -O2 hello_kernel.cu -o hello_kernel && ./hello_kernel


## 5) Testy pamięci: `cudaMalloc`, `cudaMemcpy`
Prosty test alokacji pamięci na urządzeniu oraz kopiowania Host↔Device.


In [ ]:

%%writefile memcpy_test.cu
#include <cstdio>
#include <cuda_runtime.h>

#define CUDA_CHECK(x) do {     cudaError_t err__ = (x);     if (err__ != cudaSuccess) {         std::fprintf(stderr, "CUDA error %s at %s:%d
", cudaGetErrorString(err__), __FILE__, __LINE__);         std::exit(1);     } } while(0)

int main(){
    int h = 42, h2 = 0; 
    int *d = nullptr;

    CUDA_CHECK(cudaMalloc(&d, sizeof(int)));
    CUDA_CHECK(cudaMemcpy(d, &h, sizeof(int), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(&h2, d, sizeof(int), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d));

    std::printf("Wartość po rund-trippie: %d
", h2);
    return 0;
}


In [ ]:
!nvcc -O2 memcpy_test.cu -o memcpy_test && ./memcpy_test


## 6) Strumienie CUDA — krótkie demo
Tworzymy i niszczymy strumień oraz wykonujemy prosty transfer asynchroniczny.


In [ ]:

%%writefile streams_demo.cu
#include <cstdio>
#include <cuda_runtime.h>

#define CUDA_CHECK(x) do {     cudaError_t err__ = (x);     if (err__ != cudaSuccess) {         std::fprintf(stderr, "CUDA error %s at %s:%d
", cudaGetErrorString(err__), __FILE__, __LINE__);         std::exit(1);     } } while(0)

int main(){
    cudaStream_t s; CUDA_CHECK(cudaStreamCreate(&s));
    const size_t N = 1024;
    int *d = nullptr; int *h = (int*)malloc(N*sizeof(int));
    for (size_t i=0;i<N;++i) h[i] = (int)i;

    CUDA_CHECK(cudaMalloc(&d, N*sizeof(int)));
    CUDA_CHECK(cudaMemcpyAsync(d, h, N*sizeof(int), cudaMemcpyHostToDevice, s));
    CUDA_CHECK(cudaStreamSynchronize(s));

    CUDA_CHECK(cudaFree(d));
    free(h);
    CUDA_CHECK(cudaStreamDestroy(s));
    std::puts("OK: strumień utworzony, transfer async wykonany.");
    return 0;
}


In [ ]:
!nvcc -O2 streams_demo.cu -o streams_demo && ./streams_demo


## 7) Pomiar czasu: `cudaEvent`
Mierzymy czas kopiowania Host→Device dla bufora ~64 MB.


In [ ]:

%%writefile events_timing.cu
#include <cstdio>
#include <cuda_runtime.h>

#define CUDA_CHECK(x) do {     cudaError_t err__ = (x);     if (err__ != cudaSuccess) {         std::fprintf(stderr, "CUDA error %s at %s:%d
", cudaGetErrorString(err__), __FILE__, __LINE__);         std::exit(1);     } } while(0)

int main(){
    const size_t N = (64ULL<<20)/sizeof(float); // ~64 MB
    float *h = (float*)malloc(N*sizeof(float));
    for (size_t i=0;i<N;++i) h[i] = (float)i;
    float *d = nullptr; CUDA_CHECK(cudaMalloc(&d, N*sizeof(float)));

    cudaEvent_t start, stop; CUDA_CHECK(cudaEventCreate(&start)); CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));
    CUDA_CHECK(cudaMemcpy(d, h, N*sizeof(float), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    float ms = 0.0f; CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));

    std::printf("Kopiowanie ~%.1f MB zajęło %.3f ms (%.2f GB/s)
", N*sizeof(float)/1e6, ms, (N*sizeof(float)/1e6)/(ms/1e3));

    CUDA_CHECK(cudaEventDestroy(start)); CUDA_CHECK(cudaEventDestroy(stop));
    CUDA_CHECK(cudaFree(d)); free(h);
    return 0;
}


In [ ]:
!nvcc -O2 events_timing.cu -o events_timing && ./events_timing


## 8) Wersje runtime i drivera
Przydaje się do diagnostyki niezgodności.


In [ ]:

%%writefile version_check.cu
#include <cstdio>
#include <cuda_runtime.h>

#define CUDA_CHECK(x) do {     cudaError_t err__ = (x);     if (err__ != cudaSuccess) {         std::fprintf(stderr, "CUDA error %s at %s:%d
", cudaGetErrorString(err__), __FILE__, __LINE__);         std::exit(1);     } } while(0)

static void print_ver(const char* what, int v){
    int major = v / 1000; int minor = (v % 1000) / 10; 
    std::printf("%s: %d (%d.%d)
", what, v, major, minor);
}

int main(){
    int drv=0, rt=0; 
    CUDA_CHECK(cudaDriverGetVersion(&drv));
    CUDA_CHECK(cudaRuntimeGetVersion(&rt));
    print_ver("Driver", drv);
    print_ver("Runtime", rt);
    return 0;
}


In [ ]:
!nvcc -O2 version_check.cu -o version_check && ./version_check


## 9) Unified Memory (Managed) — podstawowy test
Alokujemy zarządzaną pamięć, modyfikujemy na GPU i odczytujemy na CPU.


In [ ]:

%%writefile unified_memory_demo.cu
#include <cstdio>
#include <cuda_runtime.h>

#define CUDA_CHECK(x) do {     cudaError_t err__ = (x);     if (err__ != cudaSuccess) {         std::fprintf(stderr, "CUDA error %s at %s:%d
", cudaGetErrorString(err__), __FILE__, __LINE__);         std::exit(1);     } } while(0)

__global__ void add_one(int* p) { if (threadIdx.x == 0 && blockIdx.x == 0) (*p)++; }

int main(){
    int *ptr = nullptr; CUDA_CHECK(cudaMallocManaged(&ptr, sizeof(int)));
    *ptr = 7; // zapis na CPU
    int dev = 0; cudaGetDevice(&dev);
    // Prefetch (opcjonalny) — ignorujemy status jeśli niewspierane
    cudaMemPrefetchAsync(ptr, sizeof(int), dev);
    add_one<<<1,1>>>(ptr);
    if (cudaDeviceSynchronize()!=cudaSuccess) return 1; // prosto
    // Prefetch z powrotem na CPU (opcjonalnie)
    cudaMemPrefetchAsync(ptr, sizeof(int), cudaCpuDeviceId);
    cudaDeviceSynchronize();
    std::printf("Unified Memory wynik: %d
", *ptr);
    cudaFree(ptr);
    return 0;
}


In [ ]:
!nvcc -O2 unified_memory_demo.cu -o unified_memory_demo && ./unified_memory_demo


---
### Następne kroki
W kolejnym wpisie możesz rozwinąć temat: konfiguracja siatki (grid/block), layout pamięci (`AoS` vs `SoA`), różne rodzaje pamięci (`__shared__`, `__constant__`), oraz wprowadzenie do profilowania (Nsight Systems/Compute) i NVTX.
